# Data Ingestion, Cleaning & Preprocessing with Pandas
## Rabtech Academy — Task 3

**Dataset:** Global Superstore (Orders sheet), 51,290 records and 24 original columns.

**Objective:** Inspect and clean retail transaction data, handle missing values, duplicates, data types and outliers, engineer date/profit features, and export `clean_dataset.csv`.

> Keep the original raw file unchanged. The cleaning choices below are explicit and auditable.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

RAW_PATH = Path("raw_dataset.csv")
OUTPUT_PATH = Path("clean_dataset.csv")

df = pd.read_csv(RAW_PATH, low_memory=False)
print("Raw shape:", df.shape)
display(df.head())


## 1. Before-cleaning profile

In [ ]:
print("Shape:", df.shape)
print("\\nData types:")
display(df.dtypes.astype(str).to_frame("dtype"))
print("\\nMissing values:")
missing_before = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_before)
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Row ID values:", df["Row ID"].duplicated().sum())
print("Repeated Order IDs (may be legitimate multi-line orders):", df["Order ID"].duplicated().sum())
display(df[["Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]].describe().T)


## 2. Cleaning decisions

- Remove **exact duplicate rows** only. Do not remove repeated `Order ID` values because one order can contain multiple product lines.
- Trim whitespace in text columns.
- Parse `Order Date` and `Ship Date` as datetimes; convert numeric measures using coercion.
- Keep missing `Postal Code` values as missing. Postal codes are location identifiers, so inventing a mean/median value would be misleading. The field is retained as nullable text.
- Validate positive quantity and non-negative sales.
- Detect extreme values in `Sales`, `Profit`, and `Shipping Cost` using the 1.5×IQR rule. Cap those numeric measures at IQR fences and retain boolean flags so affected rows remain identifiable.


In [ ]:
clean = df.copy()
clean.columns = clean.columns.str.strip()

# Normalize text values
for col in clean.select_dtypes(include=["object"]).columns:
    clean[col] = clean[col].astype("string").str.strip()

# Convert date and numeric fields
for col in ["Order Date", "Ship Date"]:
    clean[col] = pd.to_datetime(clean[col], errors="coerce")

for col in ["Row ID", "Postal Code", "Sales", "Quantity", "Discount", "Profit", "Shipping Cost"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Remove exact duplicate records only
exact_duplicates_removed = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()

# Validate core transaction fields
invalid_core = clean["Quantity"].le(0) | clean["Sales"].lt(0) | clean["Quantity"].isna() | clean["Sales"].isna()
invalid_core_rows = int(invalid_core.sum())
clean = clean.loc[~invalid_core].copy()

# Postal code is an identifier: preserve missing values, do not impute.
clean["Postal Code"] = clean["Postal Code"].round().astype("Int64").astype("string")

print("Exact duplicates removed:", exact_duplicates_removed)
print("Rows excluded for invalid quantity/sales:", invalid_core_rows)
print("Rows after basic cleaning:", clean.shape)


## 3. Outlier detection and treatment

In [ ]:
outlier_summary = []
for col in ["Sales", "Profit", "Shipping Cost"]:
    q1 = clean[col].quantile(0.25)
    q3 = clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flag_col = f"{col} Outlier Flag"
    clean[flag_col] = (clean[col] < lower) | (clean[col] > upper)
    flagged = int(clean[flag_col].sum())
    clean[col] = clean[col].clip(lower=lower, upper=upper)
    outlier_summary.append({
        "column": col, "lower_iqr_fence": lower,
        "upper_iqr_fence": upper, "outliers_flagged": flagged
    })

display(pd.DataFrame(outlier_summary))


## 4. Feature engineering

In [ ]:
clean["Order Year"] = clean["Order Date"].dt.year.astype("Int64")
clean["Order Month"] = clean["Order Date"].dt.month.astype("Int64")
clean["Order Month Name"] = clean["Order Date"].dt.month_name()

# Profit margin = profit / sales × 100; undefined when sales is zero.
clean["Profit Margin (%)"] = np.where(
    clean["Sales"].ne(0),
    clean["Profit"] / clean["Sales"] * 100,
    np.nan
)

display(clean[["Order Date", "Order Year", "Order Month",
               "Order Month Name", "Sales", "Profit",
               "Profit Margin (%)"]].head())


## 5. After-cleaning validation

In [ ]:
print("Clean shape:", clean.shape)
print("Exact duplicate rows remaining:", clean.duplicated().sum())
print("\\nMissing values after cleaning:")
missing_after = pd.DataFrame({
    "missing_count": clean.isna().sum(),
    "missing_percent": (clean.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_after)
print("\\nFinal data types:")
display(clean.dtypes.astype(str).to_frame("dtype"))


## 6. Before-and-after comparison

In [ ]:
comparison = pd.DataFrame({
    "before_rows": [len(df)],
    "after_rows": [len(clean)],
    "before_columns": [df.shape[1]],
    "after_columns": [clean.shape[1]],
    "before_exact_duplicates": [int(df.duplicated().sum())],
    "after_exact_duplicates": [int(clean.duplicated().sum())],
    "before_missing_cells": [int(df.isna().sum().sum())],
    "after_missing_cells": [int(clean.isna().sum().sum())]
})
display(comparison)


## 7. Export cleaned dataset

In [ ]:
clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(clean):,} rows and {len(clean.columns)} columns to: {OUTPUT_PATH.resolve()}")


## 8. Conclusion

The notebook ingests the 51,290-row retail dataset, profiles it, normalizes text, converts dates and numeric fields, removes exact duplicate rows, validates core numeric fields, flags and caps IQR outliers, preserves missing postal codes without fabricating values, engineers year/month/profit-margin features, and exports `clean_dataset.csv`.

**Important interpretation:** Repeated order IDs are valid order-line records, not duplicates. Missing postal codes remain null because postal codes are identifiers and should not be statistically imputed.
